# 🛰️ Space Debris Detection: EDA & Model Training

This notebook walks through the data exploration and training process for the Space Debris AI object detection model. It covers:
1. Dataset structure and statistics
2. Visualizing ground truth bounded boxes
3. Launching YOLOv8 training
4. Evaluating model performance metrics

## 1. Setup & Imports

In [ ]:
import os
import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('dark_background')
import warnings
warnings.filterwarnings('ignore')

## 2. Dataset Exploration
The dataset consists of synthetically generated space images with orbital debris objects (fragments, spherical objects, and streaks). It's formatted in YOLO standard structure.

In [ ]:
DATASET_PATH = Path("../dataset")

# Load dataset configuration
with open(DATASET_PATH / "dataset.yaml", "r") as f:
    data_config = yaml.safe_load(f)

print("Classes:", data_config['names'])

splits = ['train', 'valid', 'test']
stats = []

for split in splits:
    img_dir = DATASET_PATH / split / "images"
    lbl_dir = DATASET_PATH / split / "labels"
    
    if img_dir.exists():
        num_images = len(list(img_dir.glob("*.jpg")))
        num_labels = len(list(lbl_dir.glob("*.txt")))
        stats.append({"Split": split, "Images": num_images, "Labels": num_labels})

df_stats = pd.DataFrame(stats)
display(df_stats)

### Bounding Box Distribution
Let's look at how many objects typically appear in a single image.

In [ ]:
train_lbl_dir = DATASET_PATH / "train" / "labels"
box_counts = []

if train_lbl_dir.exists():
    for txt_file in train_lbl_dir.glob("*.txt"):
        with open(txt_file, "r") as f:
            lines = f.readlines()
            box_counts.append(len(lines))

    plt.figure(figsize=(10, 5))
    plt.hist(box_counts, bins=15, color='#00b4ff', edgecolor='black')
    plt.title("Distribution of Debris Objects per Image (Train Split)")
    plt.xlabel("Number of Objects")
    plt.ylabel("Image Count")
    plt.grid(alpha=0.2)
    plt.show()

## 3. Visualizing Ground Truth Data
Let's plot a few images with their bounding boxes to verify the annotations.

In [ ]:
def draw_yolo_boxes(img_path, lbl_path):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    if lbl_path.exists():
        with open(lbl_path, "r") as f:
            for line in f:
                class_id, xc, yc, bw, bh = map(float, line.strip().split())
                x1 = int((xc - bw/2) * w)
                y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w)
                y2 = int((yc + bh/2) * h)
                
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 128), 2)
    return img

train_img_dir = DATASET_PATH / "train" / "images"
if train_img_dir.exists():
    sample_images = list(train_img_dir.glob("*.jpg"))[:4]
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for ax, img_path in zip(axes, sample_images):
        lbl_path = train_lbl_dir / (img_path.stem + ".txt")
        img_with_boxes = draw_yolo_boxes(img_path, lbl_path)
        ax.imshow(img_with_boxes)
        ax.axis('off')
        ax.set_title(img_path.name)
    plt.tight_layout()
    plt.show()

## 4. Model Training
We use **YOLOv8 Nano**, the fastest variant suitable for real-time dashboard inference. Training runs for 30-50 epochs. 

*Note: In production, you would run the training script directly via CLI for better memory management.*

In [ ]:
from ultralytics import YOLO

# Initialize model
model_name = "yolov8n.pt"
print(f"Initializing {model_name}...")
# model = YOLO(model_name)

# Example training command (uncomment to run in notebook)
# results = model.train(
#     data=str(DATASET_PATH / "dataset.yaml"),
#     epochs=30,
#     imgsz=640,
#     batch=16,
#     optimizer="AdamW",
#     project="../models",
#     name="debris_detection"
# )

## 5. Evaluation Metrics
Once training is complete, YOLOv8 generates comprehensive evaluation plots in the output directory. Let's analyze the `training_metrics.json` if available.

In [ ]:
import json

metrics_file = Path("../logs/training_metrics.json")

if metrics_file.exists():
    with open(metrics_file, "r") as f:
        metrics = json.load(f)
        
    eval_data = metrics.get("evaluation", {})
    if eval_data:
        print("\n🏆 Model Evaluation Metrics:")
        print("="*30)
        for k, v in eval_data.items():
            print(f"{k:12s}: {v:.4f}")
        print("="*30)
    else:
        print("Metrics file found but 'evaluation' data is missing.")
else:
    print("Metrics file not found. Ensure training has completed first.")

You can now use the deployed Streamlit dashboard located at `dashboard/app.py` to interactively detect debris using the trained weights (`models/debris_detector.pt`).